<a href="https://colab.research.google.com/github/project-ida/arpa-e-experiments/blob/main/templates/waveform-extraction-starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<a href="https://nbviewer.org/github/project-ida/arpa-e-experiments/blob/main/templates/waveform-extraction-starter.ipynb" target="_parent"><img src="https://nbviewer.org/static/img/nav_logo.svg" alt="Open In nbviewer" width="100"/></a>

‼️ **Prerequisites** ‼️:
- Access to the `Nucleonics` Google Drive shortcut and its `.env` folder for SQL credentials
- Access to Nucleonics data

ROOT files are downloaded from Nucleonics and saved locally for analysis. Google Drive is used only for SQL credentials.

# Waveform extraction

This notebook contains starter code that extracts and visualises event waveforms from a ROOT file.

In [ ]:
# Makes Libs accessible and runs notebook from same location regardless of whether colab or local
!pip install colocal -q
import colocal
root, branch, cwd = colocal.setup()

## Libraries

In [ ]:
# Install the ROOT reader and HTTP download client
%pip install -q uproot requests

# SQL credentials are still loaded from Google Drive
import shutil
from google.colab import drive

# Data analysis and plotting
import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt

# Our HTTP clients
from libs.sql_api import SQLConnection, SQLAPIError, read_sql
from libs.root_files_api import download_root_file

## Authentication

In [ ]:
# Mount Drive to load SQL credentials.
drive.mount('/content/drive')
shutil.copy(
    "/content/drive/MyDrive/Nucleonics/.env/psql_credentials_readonly.py",
    "psql_credentials.py",
)

In [ ]:
# Import SQL credentials
from psql_credentials import PGHOST, PGUSER, PGPASSWORD, PGPORT, PGDATABASE

# Create the database connection string
connection_uri = f'https://{PGHOST}/sql-api/query.php'
engine = SQLConnection(
    connection_uri,
    PGUSER,
    PGPASSWORD,
    database=PGDATABASE,
    timeout=(10, 600),
)

## Fetch the ROOT file corresponding to the a time period

For this notebook we must specify a:
- channel number
- time of interest
- duration around the time of interest

In [ ]:
channel_number = 6
time = '2025-09-03 12:00:00'
duration = 600

We now look in the database for the closest ROOT file to the `time`. Since our ROOT files are organised by their end time (time of the last event), we need to look for the closest ROOT file after our specified `time`.

In [ ]:
def find_root_file(event_time, channel_number):
    query = """
    SELECT *
    FROM root_files
    WHERE time > %(event_time)s
      AND file LIKE %(file_pattern)s
    ORDER BY ABS(EXTRACT(EPOCH FROM (%(event_time)s - time)))
    LIMIT 1;
    """
    file_pattern = f"%_CH{channel_number}@%"
    df = read_sql(query, con=engine, params={
        "event_time": event_time,
        "file_pattern": file_pattern
    })
    if df.empty:
        raise FileNotFoundError(f"No indexed ROOT file found after {event_time} for channel {channel_number}.")
    df.set_index('time', inplace=True)
    return df

In [ ]:
df_root = find_root_file(time, channel_number)
df_root

Let's create a dictionary for easy access to the file info:

In [ ]:
closest_root_file = df_root.iloc[0].to_dict()

Download the selected ROOT file from Nucleonics. It will be saved under `downloads/root_files` for the analysis below.

In [ ]:
ROOT_FILES_BASE_URL = "http://nucleonics.mit.edu:8080"
DOWNLOAD_DIR = "downloads/root_files"


def show_download_progress(received, total):
    downloaded_mb = received / 1024**2
    if total:
        print(f"Download {100 * received / total:.1f}% ({downloaded_mb:.1f} MB)", end="\r")
    else:
        print(f"Downloaded {downloaded_mb:.1f} MB", end="\r")

In [ ]:
print(f"Downloading {closest_root_file['file']} from Nucleonics...")

In [ ]:
filename = download_root_file(
    closest_root_file,
    download_dir=DOWNLOAD_DIR,
    base_url=ROOT_FILES_BASE_URL,
    progress=show_download_progress,
)
print(f"\nSaved to {filename}")

## Analysing waveform data

### Pulses with absolute time

Each radiation pulse is stored in the ROOT files with a `Timestamp`, `Energy`, `EnergyShort` and `Samples`. The waveform of the pulse is stored in the `Samples` part of the ROOT tree.

`Timestamp` information measures the number of picoseconds from the start of the experiment. In order for us to conveniently extract pulses using absolute time, we will need to create a dataframe with the appropriate time index.

In [ ]:
# Opens up the ROOT file and extracts the information
with uproot.open(filename) as f:
    tree = f['Data_R']
    branches = tree.keys()
    ts = tree['Timestamp'].array(library='np')
    e = tree['Energy'].array(library='np')
    wf = tree['Samples'].array(library='np')

    # Conditionally load EnergyShort if it exists
    if 'EnergyShort' in branches:
        es = tree['EnergyShort'].array(library='np')
    else:
        es = None

# Creates dataframe with absolute time index
end_time = df_root.index[0]
start_time = end_time - pd.to_timedelta((ts[-1]-ts[0])/1e12, unit='s')
time_abs = start_time + pd.to_timedelta((ts - ts[0])/1e12, unit='s')

data = {'Energy': e, 'Waveform': wf}
if es is not None:
    data['EnergyShort'] = es
df_pulses = pd.DataFrame(data, index=pd.to_datetime(time_abs))

In [ ]:
df_pulses.head()

### Filtering pulses

We don't necessarily want to see all pulses. We might wish to only see high energy pulses, or filter out neutrons from gammas (if using an Eljen detector) using the PSP value (More discussion about neutron/gamma discrimination can be found in the [PSD Analysis notebook](https://github.com/project-ida/arpa-e-experiments/blob/main/tutorials/PSD-analysis.ipynb)).

PSP is calculated by:

$$
\rm PSP = \frac{\rm Energy - EnergyShort}{\rm Energy}
$$



In [ ]:
if "EnergyShort" in df_pulses.columns:
  df_pulses['PSP'] = 1 - (df_pulses['EnergyShort'] / df_pulses['Energy'])

In [ ]:
psp_range = [0,1] # All PSP values

energy_range = [0,70] # low energy
# energy_range = [71,600] # high energy
# energy_range = [901,3100] # very high energy

In [ ]:
filtered = df_pulses[(df_pulses['Energy'] > energy_range[0]) & (df_pulses['Energy'] < energy_range[1])]

if "PSP" in df_pulses.columns:
  filtered = filtered[(filtered['PSP'] > psp_range[0]) & (filtered['PSP'] < psp_range[1])]

### Visualising the counts

Let's now do a sanity check and visualise the counts per second over the entire ROOT file. The plot below should look identical to what we see in our data panels.

We'll also mark the burst time:

In [ ]:
time = pd.to_datetime(time)
print(time)

In [ ]:
cps = filtered["Energy"].resample('1s').count()
plt.figure(figsize=(12, 4))
plt.plot(cps, drawstyle='steps-mid')
plt.axvline(time, color='r', linestyle='--', label="Time of interest")
plt.title('Full ROOT file')
plt.xlabel('Time')
plt.ylabel('Counts per second')
plt.legend()
plt.grid(True)
plt.show()

Let's now zoom in on the time period of interest which is around:

In [ ]:
print(time)

with a duration of:

In [ ]:
duration = pd.to_timedelta(duration, unit='s')
print(duration)

Once we determine the `start` and `end` times, we can easily pull out cps and waveforms for the burst.

In [ ]:
start = time - duration/2
end = time + duration/2

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(cps[start:end], drawstyle='steps-mid', color='orange')
plt.axvline(time, color='r', linestyle='--', label="Time of interest")
plt.title('Period of interest')
plt.xlabel('Time')
plt.ylabel('Counts per second')
plt.legend()
plt.grid(True)
plt.show()

### Visualising the waveforms

We can now easily extract and visualise the waveforms using the `start` and `end` times.

In [ ]:
df_period = filtered[start:end]

plt.figure(figsize=(10, 6))
for wf in df_period['Waveform']:
    plt.plot(wf, alpha=0.5)

if "EnergyShort" in df_pulses.columns:
  plt.title(f'Waveforms for pulses between {start} and {end} \n with PSP in range {psp_range}\n and energy in range {energy_range}')
else:
  plt.title(f'Waveforms for pulses between {start} and {end} \n with energy in range {energy_range}')
plt.xlabel('Sample number')
plt.ylabel('Amplitude')
plt.grid(True)
plt.show()